### Cheks of basic SMC (save my chatbot browser plugin) file properties

In [ ]:
import pathlib as pl
import sys
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

In [2]:
def is_smc_file(file_path):
    # test harness function
    try:
        content = lpz.read_markdown_file(file_path)
    except Exception as e:
        raise Exception(e)
    
    try:
        lpz.is_smc_content(content)
    except Exception as e:
        raise Exception(e)

In [3]:
datdir = rfw.refwrangle_test_dir / 'dat'

file_path_perp = datdir / "perplexity_example.md"
file_path_smc = datdir / "perplexity_single_prompt_savemychatbot_example.md"
file_path_not_md = datdir / 'obsnotecitekeys.csv'
file_path_no_exist = datdir / "__alsdkfjasd__.md"

ic(is_smc_file(file_path_perp))
ic(is_smc_file(file_path_smc))
try:
    ic(is_smc_file(file_path_not_md))
except:
    print("Not md file exception, as expected")
try:
    ic(is_smc_file(file_path_no_exist))
except:
    print("Non-existent file exception, as expected")

ic| is_smc_file(file_path_perp): None
ic| is_smc_file(file_path_smc): None


Not md file exception, as expected
Non-existent file exception, as expected


In [4]:
# Example usage
file_contents_1 = """
## User
What is the weather today?
## AI Answer
It's sunny and warm.
"""

file_contents_2 = """
## User
What is the weather today?
## AI Answer
It's sunny and warm.
## User
Tell me a joke.
## AI Answer
Why don’t skeletons fight each other? They don’t have the guts.
"""

file_contents_unmatched = """
## User
What is the weather today?
## AI Answer
It's sunny and warm.
## User
Tell me a joke.
"""

file_contents_misordered = """
## AI Answer
What is the weather today?
## User
It's sunny and warm.
"""

ic(lpz.count_prompts_smc_content(file_contents_1))
ic(lpz.count_prompts_smc_content(file_contents_2))
try:
    ic(lpz.count_prompts_smc_content(file_contents_unmatched))
except:
    print('Threw exception for incomplete pair, as it should')
try:    
    ic(lpz.count_prompts_smc_content(file_contents_misordered))
except:
    print('Threw exception for misordered pair, as it should')    

ic| lpz.count_prompts_smc_content(file_contents_1): 1
ic| lpz.count_prompts_smc_content(file_contents_2): 2


Threw exception for incomplete pair, as it should
Threw exception for misordered pair, as it should


In [ ]:
import re
from collections import Counter
from typing import List, Tuple
from tmp2 import read_markdown_file, is_smc_content, count_prompts_smc_content


def extract_prompt(file_contents: str) -> str:
    """Extracts the prompt text between ## User and ## AI Answer."""
    lines = file_contents.splitlines()
    user_prompt = []
    inside_user_section = False

    for line in lines:
        line = line.strip()
        if line == "## User":
            inside_user_section = True
            continue
        elif line == "## AI Answer":
            break
        elif inside_user_section:
            user_prompt.append(line)

    if not user_prompt:
        raise ValueError("No prompt found between ## User and ## AI Answer headers.")
    
    return " ".join(user_prompt).strip()


def are_prompts_equivalent(prompt1: str, prompt2: str) -> bool:
    """
    Compares two prompts for equivalence in meaning.
    For simplicity, this function uses case-insensitive string comparison.
    """
    return prompt1.strip().lower() == prompt2.strip().lower()


def find_most_representative_prompt(prompts: List[str]) -> str:
    """
    Finds the most representative prompt from a list of prompts.
    This implementation returns the most common prompt.
    
    If there's a tie in frequency, it returns the first one encountered.
    """
    counter = Counter(prompts)
    most_common_prompt, _ = counter.most_common(1)[0]
    return most_common_prompt


def verify_files(file_paths: List[str]) -> Tuple[bool, str]:
    """
    Verifies that all files are valid SMC markdown files, contain one prompt each,
    and have identical prompts. Returns True and the most representative prompt if valid.
    
    :param file_paths: List of file paths to verify
    :return: Tuple (is_valid, representative_prompt)
             - is_valid: True if all files pass validation, False otherwise
             - representative_prompt: The most representative prompt if valid, None otherwise
    """
    prompts = []

    for file_path in file_paths:
        # Step 1: Read and validate SMC markdown content
        try:
            content = read_markdown_file(file_path)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
            return False, None

        if not is_smc_content(content):
            print(f"File {file_path} is not a valid SMC markdown file.")
            return False, None

        # Step 2: Ensure there is exactly one prompt
        try:
            prompt_count = count_prompts_smc_content(content)
            if prompt_count != 1:
                print(f"File {file_path} does not contain exactly one prompt.")
                return False, None
        except Exception as e:
            print(f"Error counting prompts in file {file_path}: {e}")
            return False, None

        # Step 3: Extract and store the prompt
        try:
            prompt = extract_prompt(content)
            prompts.append(prompt)
        except Exception as e:
            print(f"Error extracting prompt from file {file_path}: {e}")
            return False, None

    # Step 4: Check if all prompts have the same meaning
    first_prompt = prompts[0]
    for i, prompt in enumerate(prompts[1:], start=2):
        if not are_prompts_equivalent(first_prompt, prompt):
            print(f"Prompts in files do not have identical meanings (File {i}).")
            return False, None

    # Step 5: Identify the most representative prompt
    most_representative_prompt = find_most_representative_prompt(prompts)

    return True, most_representative_prompt


# Test Cases
def test_verify_files():
    """Test cases for verifying files."""
    
    # Test Case 1: All valid files with identical prompts
    file_paths_1 = ["test_files/file1.md", "test_files/file2.md", "test_files/file3.md"]
    
    is_valid_1, representative_prompt_1 = verify_files(file_paths_1)
    
    assert is_valid_1 is True, "Test Case 1 Failed"
    assert representative_prompt_1 == "What is the capital of France?", "Test Case 1 Failed"
    
    # Test Case 2: Files with different prompts
    file_paths_2 = ["test_files/file4.md", "test_files/file5.md"]
    
    is_valid_2, representative_prompt_2 = verify_files(file_paths_2)
    
    assert is_valid_2 is False, "Test Case 2 Failed"
    
    # Test Case 3: Invalid SMC markdown file
    file_paths_3 = ["test_files/invalid_file.md"]
    
    is_valid_3, representative_prompt_3 = verify_files(file_paths_3)
    
    assert is_valid_3 is False, "Test Case 3 Failed"
    
    print("All test cases passed!")


# Run Tests
if __name__ == "__main__":
    test_verify_files()
